In [4]:
import os

saved_model_path = "./yolov1_tensorflow"

# 파일이 남아있으면 지우기
if os.path.isfile(saved_model_path):
    os.remove(saved_model_path)

# 폴더 생성
os.makedirs(saved_model_path, exist_ok=True)


In [5]:
import onnx2tf

onnx_path = "yolov1.onnx"

onnx2tf.convert(
    input_onnx_file_path=onnx_path,
    output_folder_path=saved_model_path,
    non_verbose=True
)

In [9]:
import tensorflow as tf

saved_model_path = "./yolov1_tensorflow"
tflite_ready_path = "./yolov1_saved_model_tflite"

# SavedModel 불러오기
model = tf.saved_model.load(saved_model_path)

# 시그니처 생성
@tf.function(input_signature=[tf.TensorSpec([1, 448, 448, 3], tf.float32, name="input")])
def serve_fn(input):
    return model(input)

# 저장
tf.saved_model.save(model, tflite_ready_path, signatures={"serving_default": serve_fn})


In [10]:
import tensorflow as tf
import numpy as np
from PIL import Image
import os

img_dir = "./VOCdevkit/VOC2007/JPEGImages"
img_files = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.endswith(".jpg")]

def ResizePreprocess(img, size=(448, 448)):
    # 이미지 리사이즈(0~1 스케일링)
    img = img.resize(size)
    img = np.array(img).astype(np.float32) / 255.0  # float32, 0~1
    img = np.expand_dims(img, axis=0)  # 배치 차원 추가 [1, 448, 448, 3]
    return img

def representative_dataset():
    for img_path in img_files[:100]:  
        img = Image.open(img_path).convert("RGB")
        img = ResizePreprocess(img)
        yield [img]

converter = tf.lite.TFLiteConverter.from_saved_model("./yolov1_saved_model_tflite")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
converter.inference_input_type = tf.int8 
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

tflite_model_file = "yolov1_int8.tflite"
with open(tflite_model_file, "wb") as f:
    f.write(tflite_model)

print(f"TFLite int8 model save {tflite_model_file}")    

TFLite int8 model save yolov1_int8.tflite
